In [1]:
import os
os.environ["HF_HOME"] = "/media/volume/AI_Core_CoDIRA_StorageVolume/huggingface_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import pipeline, BitsAndBytesConfig
import torch

def main():
    # Verify GPU is available
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU found, running on CPU (will be very slow)")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    print("Loading model onto GPU...")
    pipe = pipeline(
        "text-generation",
        model="google/medgemma-27b-text-it",
        model_kwargs={
            "quantization_config": quantization_config,
            "dtype": torch.bfloat16,
        },
        device_map="cuda:0",   # force GPU instead of "auto"
    )
    print("Model loaded!")

    messages = [
        {"role": "system", "content": "You are a warm, empathetic GP. Use plain language."},
        {"role": "user", "content": "I'm having stomach pain. What should I do?"}
    ]

    print("Generating response...")
    output = pipe(messages, max_new_tokens=200)  # fixed: max_new_tokens not max_length
    print("\n--- Response ---")
    print(output[0]["generated_text"][-1]["content"])

if __name__ == "__main__":
    main()

CUDA available: True
GPU: GRID A100X-20C
Loading model onto GPU...


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded!
Generating response...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Response ---
Oh dear, I'm sorry to hear you're having stomach pain. That's never a nice feeling. Let's try and figure out what might be going on and what you can do.

First, can you tell me a bit more about the pain? Things like:

*   **Where exactly is the pain?** Is it all over, or in one specific spot (like upper right, lower left, around your belly button)?
*   **What does the pain feel like?** Is it sharp, dull, crampy, burning, like pressure?
*   **When did it start?** Was it sudden or did it come on gradually?
*   **Is it constant, or does it come and go?** If it comes and goes, does anything make it better or worse (like eating, lying down, moving around)?
*   **Have you had any other symptoms?** Like feeling sick (nausea), actually being sick


In [ ]:
import pandas as pd
from datasets import Dataset

try:
    # Read JSON file (adjust orient/lines if needed based on JSON structure)
    df = pd.read_json('/home/exouser/Desktop/Emergency-Room-Simulator-Repository/rag_backend/finetuning/data/medgemma_finetuning_data.json')
    dataset = Dataset.from_pandas(df)
    print("Dataset loaded successfully")
    print(dataset)
except FileNotFoundError:
    print("Please upload 'medgemma finetuning data'")

Dataset loaded successfully
Dataset({
    features: ['messages'],
    num_rows: 3
})


: 

In [5]:
import torch
print(torch.__version__)        # should be 2.5.1+cu121
print(torch.version.cuda)       # should be 12.1
print(torch.cuda.is_available()) # should be True

2.5.1+cu121
12.1
True


In [3]:

import os
os.environ["HF_HOME"] = "/media/volume/AI_Core_CoDIRA_StorageVolume/huggingface_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/medgemma-27b-text-it"

# 4-bit Quantization Config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa"
)


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

In [7]:
model = prepare_model_for_kbit_training(model)

OutOfMemoryError: CUDA out of memory. Tried to allocate 5.25 GiB. GPU 0 has a total capacity of 20.00 GiB of which 2.57 GiB is free. Including non-PyTorch memory, this process has 15.75 GiB memory in use. Of the allocated memory 15.10 GiB is allocated by PyTorch, and 179.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()